In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Import Libraries

In [ ]:
pip install transformers datasets accelerate

In [ ]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

from sklearn.model_selection import train_test_split

In [ ]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)

In [ ]:
wandb.init(
    project="24f3004524-t22026",
    name="deberta-baseline-run1"
)

# EDA

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

In [ ]:
train.head()

In [ ]:
train['answer'].value_counts()

In [ ]:
train.isnull().sum()

# Evaluation Metric

In [ ]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    
    predictions = np.argsort(logits, axis=-1)[:, ::-1][:, :3]
    
    score = 0.0
    for actual, pred in zip(labels, predictions):
        if actual == pred[0]:
            score += 1.0
        elif actual == pred[1]:
            score += 0.5
        elif actual == pred[2]:
            score += 1/3
            
    return {"map3": score / len(labels)}

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Model 2 (Pretrained): DeBERTa

In [ ]:
train_questions, val_questions = train_test_split(
    train,
    test_size=0.2,
    stratify=train["answer"],
    random_state=4524
)

print(train_questions.shape)
print(val_questions.shape)

In [ ]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess_multiple_choice(examples):
    first_sentences = [[prompt] * 5 for prompt in examples["prompt"]]
    
    choices = ["A", "B", "C", "D", "E"]
    second_sentences = [
        [examples[choice][i] for choice in choices]
        for i in range(len(examples["prompt"]))
    ]
    
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    tokenized = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=384, 
        padding="max_length"
    )
    
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

choice_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_questions["label"] = train_questions["answer"].map(choice_map)
val_questions["label"] = val_questions["answer"].map(choice_map)

train_ds = Dataset.from_pandas(train_questions)
val_ds = Dataset.from_pandas(val_questions)

train_ds = train_ds.map(preprocess_multiple_choice, batched=True)
val_ds = val_ds.map(preprocess_multiple_choice, batched=True)

In [ ]:
train_ds = train_ds.remove_columns(["prompt", "A", "B", "C", "D", "E", "answer"])
val_ds = val_ds.remove_columns(["prompt", "A", "B", "C", "D", "E", "answer"])

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
model = AutoModelForMultipleChoice.from_pretrained(model_name,torch_dtype=torch.float32)
model = model.cuda()

In [ ]:
training_args = TrainingArguments(
    output_dir="./deberta_base",
    learning_rate=1e-5,
    per_device_train_batch_size=2,  
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  
    gradient_checkpointing=True,    
    num_train_epochs=10,
    logging_steps=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,    
    metric_for_best_model="map3", 
    greater_is_better=True,
    fp16=True,                      
    bf16=False,
    report_to="wandb"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

In [ ]:
test_ds = Dataset.from_pandas(test)
test_ds = test_ds.map(preprocess_multiple_choice, batched=True)
test_ds = test_ds.remove_columns(["prompt", "A", "B", "C", "D", "E"]) 
test_ds.set_format("torch", columns=["input_ids", "attention_mask"])

test_preds = trainer.predict(test_ds)

logits = test_preds.predictions
top3_indices = np.argsort(logits, axis=-1)[:, ::-1][:, :3]

choices = ["A", "B", "C", "D", "E"]
test_predictions = [" ".join([choices[i] for i in row]) for row in top3_indices]

# Submission Cell

In [ ]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()